In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

In [ ]:
data = pd.read_csv('./train.csv')
y = data['Survived']
passenger_id = data['PassengerId']
data.drop(['Survived', 'Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(data, y, test_size=0.2, random_state=39)

In [ ]:
numerical_columns = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X_train.select_dtypes(include=["category", "object"]).columns
preprocessor = ColumnTransformer([
  ('num', MinMaxScaler(), numerical_columns),
  ('cat', OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

In [ ]:
preprocessor

In [ ]:
param_grid = {
    "classifier__n_estimators": [100, 300, 500],
    "classifier__max_depth": [2, 3, 4, 5, 6],
    "classifier__min_samples_split": [2, 3, 4, 5],
    "classifier__min_samples_leaf": [1, 2, 3, 4, 5]
}

grid_search = GridSearchCV(
  estimator=RandomForestClassifier(),
  cv=3,
  n_jobs=-1,
  param_grid=param_grid
)

grid_search.fit(X_train, y_train)

In [ ]:
classifier = RandomForestClassifier(
  max_depth=6,
  min_samples_leaf=3,
  min_samples_split=2,
  n_estimators=1000, 
)

model = Pipeline([
  ('process', preprocessor),
  ('classifier', classifier)
])


model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_pred_cv = grid_search.predict(X_valid)
score_grid = round(accuracy_score(y_valid, y_pred_cv), 4)
print("Accuracy:", score_grid)

y_pred_model = model.predict(X_valid)
score_model = round(accuracy_score(y_valid, y_pred_model), 4)
print("Accuracy:", score_model)


In [ ]:
test_data = pd.read_csv("./test.csv")

submission_data = grid_search.predict(test_data)

# test_data[submission_data]
submission_df = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': submission_data
})
print(submission_df)
submission_df.to_csv('submission2.csv', index=False)
